In [1]:
import pandas as pd
import numpy as np
import vnstock
from vnstock import Quote
import statsmodels.api as sm
import matplotlib.pyplot as plt

from vnstock import Finance

Phiên bản Vnstock 3.5.0 đã có mặt, vui lòng cập nhật với câu lệnh : `pip install vnstock --upgrade`.
Lịch sử phiên bản: https://vnstocks.com/docs/tai-lieu/lich-su-phien-ban
Phiên bản hiện tại 3.2.6

Phiên bản Vnai 2.4.0 đã có mặt, vui lòng cập nhật với câu lệnh : `pip install vnai --upgrade`.
Lịch sử phiên bản: https://pypi.org/project/vnai/#history
Phiên bản hiện tại 2.1.9

In [2]:
start = "2007-01-01"
end = "2025-12-31"
quote1 = Quote(symbol='FPT', source='VCI')
stock = quote1.history(start=start, end=end, interval='1D')

In [3]:
stock['time'] = pd.to_datetime(stock['time'])
price_series = stock.set_index('time')['close']

price_series = price_series.sort_index()

print(price_series.head())
print("\nKiểu dữ liệu:", type(price_series))

time
2007-01-02    10.74
2007-01-03    10.98
2007-01-04    11.52
2007-01-05    12.08
2007-01-08    11.95
Name: close, dtype: float64

Kiểu dữ liệu: <class 'pandas.core.series.Series'>


In [4]:

finance = Finance(symbol='FPT', source='VCI')
df_income = finance.income_statement(period='quarterly')
print(df_income.head(10))

  ticker  yearReport  lengthReport  Revenue YoY (%)  Revenue (Bn. VND)  \
0    FPT        2025             4         0.147742     20258865670867   
1    FPT        2025             3         0.078455     17225507121683   
2    FPT        2025             2         0.092644     16658335760739   
3    FPT        2025             1         0.139902     16064980391264   
4    FPT        2024             4         0.201095     17651065378939   
5    FPT        2024             3         0.160630     15972397069700   
6    FPT        2024             2         0.221184     15245892288520   
7    FPT        2024             1         0.206306     14093297397476   
8    FPT        2023             4         0.125817     14695806584757   
9    FPT        2023             3         0.234380     13761831884948   

   Attribute to parent company (Bn. VND)  Attribute to parent company YoY (%)  \
0                          2509522220120                             0.198019   
1                      

In [5]:

quarter_end_dates = {
    '1': '03-31', 
    '2': '06-30', 
    '3': '09-30', 
    '4': '12-31'
}

# 2. Hàm chuyển đổi dòng dữ liệu thành Datetime chuẩn
def convert_to_date(row):
    year = str(row['yearReport'])
    # Xử lý an toàn: lấy số quý (phòng trường hợp nó trả về dạng 'Q1' hoặc số nguyên 1)
    quarter = str(row['lengthReport']).replace('Q', '').strip()
    month_day = quarter_end_dates.get(quarter, '12-31')
    return pd.to_datetime(f"{year}-{month_day}")

df_income['date'] = df_income.apply(convert_to_date, axis=1)

# 4. Trích xuất thành Series cho Machine Learning
revenue_series = df_income.set_index('date')['Net Sales']

# 5. Sắp xếp lại dữ liệu theo trục thời gian (từ quá khứ đến hiện tại)
revenue_series = revenue_series.sort_index()

print(revenue_series)

date
2007-03-31     5663584718970
2007-06-30     6070562308072
2013-03-31     8611039318749
2013-06-30     9336561564148
2013-09-30    12000599396256
2013-12-31    11283426979266
2014-03-31    10867321675354
2014-06-30    10685411011162
2014-09-30    13477214908110
2014-12-31    15333191333048
2015-03-31     8641481450290
2015-06-30     9328539915897
2015-09-30     9527463506576
2015-12-31    10449042783674
2016-03-31     8466518261019
2016-06-30     8958526432579
2016-09-30     9755288350096
2016-12-31    12399573089770
2017-03-31     9418119567621
2017-06-30     9946702598780
2017-09-30    10831198037445
2017-12-31    12495737352665
2018-03-31     4750768611617
2018-06-30     5474262724100
2018-09-30     6036058933435
2018-12-31     6952446588573
2019-03-31     5666479616064
2019-06-30     6825918524598
2019-09-30     7104374137543
2019-12-31     8120187874196
2020-03-31     6630564911438
2020-06-30     6980002498337
2020-09-30     7553128504072
2020-12-31     8666704612977
2021-03-3

In [6]:

# --- 1. HÀM CUSUM TÌM SỰ KIỆN DOANH THU ĐỘT BIẾN ---

def calculate_non_uniform_lagged_change(series: pd.Series, n_days: int):
    """Tính toán % thay đổi (YoY) cho dữ liệu không đều đặn (như BCTC)"""
    _timedelta = pd.Timedelta(days=n_days)
    _idx = series.index.searchsorted(series.index - _timedelta)
    _idx = _idx[_idx > 0]
    
    _series = series.iloc[-_idx.shape[0]:]
    _pad_length = series.shape[0] - _idx.shape[0]
    _na_pad = pd.Series(None, index=series.index[:_pad_length])
    
    _lagged_series = series.iloc[_idx]
    _diff = pd.Series(_series.values - _lagged_series.values, index=_series.index)
    return pd.concat([_na_pad, _diff])

def calculate_cusum_events(series: pd.Series, filter_threshold: float) -> pd.DatetimeIndex:
    """Bộ lọc CUSUM đối xứng"""
    event_dates = list()
    s_up, s_down = 0, 0
    for date, price in series.items():
        s_up = max(0, s_up + price)
        s_down = min(0, s_down + price)
        if s_up > filter_threshold:
            s_up = 0
            event_dates.append(date)
        elif s_down < -filter_threshold:
            s_down = 0
            event_dates.append(date)
    return pd.DatetimeIndex(event_dates)

def get_revenue_events(revenue_series: pd.Series, threshold=0.1):
    
    series_log = np.log(revenue_series.astype(float)) # Chuyển sang Log
    # So sánh với 360 ngày trước (khoảng 1 năm - YoY)
    series_diff = calculate_non_uniform_lagged_change(series_log, 360) 
    return calculate_cusum_events(series_diff.dropna(), threshold)

In [7]:
# --- 2. HÀM DÁN NHÃN TRIPLE BARRIER ---

def compute_triple_barrier(price_series: pd.Series, event_index: pd.DatetimeIndex, 
                           time_delta_days=90, upper_z=2.0, lower_z=-2.0, vol_span=20):
    """Tính toán nhãn 1, 0, -1 theo phương pháp 3 rào chắn"""
    timedelta = pd.Timedelta(days=time_delta_days)
    series_log = pd.Series(np.log(price_series.astype(float).values), index=price_series.index)
    
    labels, label_dates = [], []
    
    # Tính độ biến động (Volatility) trượt 20 ngày
    volatility = series_log.ewm(span=vol_span).std()
    volatility *= np.sqrt(time_delta_days / vol_span)
    
    for event_date in event_index:
        # Nếu ngày sự kiện không có trong dữ liệu giá, tìm ngày giao dịch gần nhất sau đó
        if event_date not in series_log.index:
            valid_dates = series_log.index[series_log.index >= event_date]
            if len(valid_dates) == 0: continue
            event_date = valid_dates[0]

        date_barrier = event_date + timedelta
        start_price = series_log.loc[event_date]
        
        # Dữ liệu giá trong khoảng thời gian rào chắn (90 ngày)
        log_returns = series_log.loc[event_date:date_barrier] - start_price
        
        candidates = []
        upper_barrier = upper_z * volatility.loc[event_date]
        lower_barrier = lower_z * volatility.loc[event_date]
        
        # Kiểm tra chạm rào trên (Chốt lời)
        date_upper = log_returns[log_returns > upper_barrier].first_valid_index()
        if date_upper: candidates.append((1, date_upper))
            
        # Kiểm tra chạm rào dưới (Cắt lỗ)
        date_lower = log_returns[log_returns < lower_barrier].first_valid_index()
        if date_lower: candidates.append((-1, date_lower))
            
        if candidates:
            # Lấy sự kiện xảy ra sớm nhất
            label, label_date = min(candidates, key=lambda x: x[1])
        else:
            # Không chạm rào nào -> Hết thời gian
            label, label_date = 0, log_returns.index[-1] if len(log_returns) > 0 else date_barrier
            
        labels.append(label)
        label_dates.append(label_date)
        
    return pd.Series(labels, index=event_index), pd.Series(label_dates, index=event_index)

In [8]:
# --- 3. KIỂM THỬ TRÊN DỮ LIỆU THỰC TẾ ---

print("Bắt đầu chạy thuật toán...\n")

# 1. Tìm các ngày có sự kiện biến động doanh thu
events = get_revenue_events(revenue_series, threshold=0.1)
print(f"Số lượng sự kiện CUSUM tìm thấy: {len(events)}")

# Lọc bỏ các sự kiện xảy ra trong 90 ngày gần đây nhất (vì chưa đủ thời gian để biết kết quả rào chắn)
cutoff_date = price_series.index.max() - pd.Timedelta(days=90)
events = events[events <= cutoff_date]

# 2. Dán nhãn
if len(events) > 0:
    # upper_z=2.0 và lower_z=-2.0 phù hợp với biên độ VN-Index
    labels, spans = compute_triple_barrier(price_series, events, time_delta_days=90, upper_z=2.0, lower_z=-2.0)
    
    print("\nPhân phối nhãn Triple Barrier:")
    print(labels.value_counts())
else:
    print("Không có sự kiện nào để dán nhãn. Cần giảm threshold xuống.")

Bắt đầu chạy thuật toán...

Số lượng sự kiện CUSUM tìm thấy: 34

Phân phối nhãn Triple Barrier:
 0    15
 1    14
-1     4
Name: count, dtype: int64


In [9]:
from scipy.stats import hmean

# --- 4. HÀM TÍNH TRỌNG SỐ ĐỘC NHẤT (UNIQUENESS) ---
def calculate_weights(event_spans: pd.Series, price_index: pd.DatetimeIndex) -> pd.Series:
    
    # Tạo ma trận nhị phân đánh dấu thời gian lệnh mở
    df = pd.DataFrame(0, index=price_index, columns=range(event_spans.shape[0]))
    
    for i, (event_start, event_end) in enumerate(event_spans.items()):
        mask = (price_index >= event_start) & (price_index <= event_end)
        df.loc[mask, df.columns[i]] = 1

    uniquenesses = []
    for i, (event_start, event_end) in enumerate(event_spans.items()):
        mask = (price_index >= event_start) & (price_index <= event_end)
        concurrency = df.loc[mask].sum(axis=1)
        
        # Tránh lỗi chia cho 0
        concurrency = concurrency[concurrency > 0] 
        if len(concurrency) > 0:
            uniqueness = 1.0 / hmean(concurrency)
        else:
            uniqueness = 1.0
        uniquenesses.append(uniqueness)

    return pd.Series(uniquenesses, index=event_spans.index)


# --- 5. HÀM TẠO ĐẶC TRƯNG (FEATURE ENGINEERING) ---
def _calc_rolling_vol(series, n):
    """Tính độ biến động chuẩn hóa theo năm"""
    return series.rolling(n).std() * np.sqrt(252 / n)

def build_features_matrix(price_series, revenue_series):
    """Tính toán 15 đặc trưng (Features) từ đa khung thời gian"""
    log_revenue = np.log(revenue_series.astype(float))
    log_prices = np.log(price_series.astype(float))

    log_prices_ma = log_prices.rolling(window=10).mean()
    log_returns = log_prices.diff()

    features_by_name = {}
    for i in [7, 30, 90, 180, 360]:
        # 1. Động lượng giá (Tính thẳng bằng hàm diff của Pandas cho nhanh)
        features_by_name[f'{i}_day_return'] = log_prices_ma.diff(periods=i)
        
        # 2. Độ biến động
        features_by_name[f'{i}_day_vol'] = _calc_rolling_vol(log_returns, i)
        
        # 3. Động lượng doanh thu (Dùng hàm lagged_change vì dữ liệu nhảy cóc theo quý)
        features_by_name[f'{i}_day_revenue_delta'] = calculate_non_uniform_lagged_change(log_revenue, i)

    return pd.DataFrame(features_by_name)


# --- LẮP RÁP BỘ DỮ LIỆU CUỐI CÙNG ---
print("\n--- BƯỚC CUỐI: TẠO DATAFRAME CHO MACHINE LEARNING ---")

# 1. Lấy spans (thời gian kết thúc) và tính weights
weights = calculate_weights(spans, price_series.index)
print(f"Trọng số độc nhất trung bình: {weights.mean():.4f}")

# 2. Xây dựng Features
features_df = build_features_matrix(price_series, revenue_series)

# 3. Chỉ lấy Features tại những ngày có sự kiện (CUSUM)
features_on_events = features_df.loc[events]

# 4. Gom Labels (y), Weights (w) và Features (X) thành 1 bảng
labels_df = pd.DataFrame(labels, columns=['y'])
weights_df = pd.DataFrame(weights, columns=['weights'])

final_ml_df = pd.concat([features_on_events, weights_df, labels_df], axis=1)

# Loại bỏ các dòng bị khuyết dữ liệu (NaN) do tính toán trượt (rolling)
final_ml_df.dropna(inplace=True)

print(f"\n✅ Đã " + "lắp ráp" + f" xong! Kích thước dữ liệu: {final_ml_df.shape}")
print(final_ml_df.head())


--- BƯỚC CUỐI: TẠO DATAFRAME CHO MACHINE LEARNING ---
Trọng số độc nhất trung bình: 0.9966

✅ Đã lắp ráp xong! Kích thước dữ liệu: (21, 17)
            7_day_return  7_day_vol  7_day_revenue_delta  30_day_return  \
2013-09-30      0.023676   0.068082                  0.0      -0.000960   
2013-12-31     -0.002011   0.041890                  0.0       0.008262   
2014-03-31      0.036032   0.119082                  0.0       0.232884   
2014-06-30      0.006256   0.033543                  0.0       0.029205   
2014-09-30     -0.046942   0.079558                  0.0       0.064834   

            30_day_vol  30_day_revenue_delta  90_day_return  90_day_vol  \
2013-09-30    0.039231                   0.0       0.137573    0.021481   
2013-12-31    0.018228                   0.0       0.033572    0.015300   
2014-03-31    0.069471                   0.0       0.422735    0.028399   
2014-06-30    0.036677                   0.0       0.064079    0.041464   
2014-09-30    0.054171           

In [10]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RepeatedKFold
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

print("🤖 BƯỚC 6: HUẤN LUYỆN MÔ HÌNH RANDOM FOREST\n")

# 1. Tách các thành phần: X (Đặc trưng), y (Nhãn dự báo), w (Trọng số)
X = final_ml_df.drop(columns=['y', 'weights'])
y = final_ml_df['y'].astype(int)
w = final_ml_df['weights']

# 2. Khởi tạo mô hình AI
clf = RandomForestClassifier(
    n_estimators=100, 
    max_depth=3, 
    class_weight='balanced_subsample', 
    random_state=42
)

# 3. Chạy Kiểm thử chéo (Cross-Validation) để xem năng lực thực sự
print("Đang chạy Cross-Validation (5-Fold, 4 Repeats)...")
rkf = RepeatedKFold(n_splits=5, n_repeats=4, random_state=42)
cv_scores = []

for train_idx, test_idx in rkf.split(X):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    w_train = w.iloc[train_idx]
    
    # Huấn luyện trên tập Train
    clf.fit(X_train, y_train, sample_weight=w_train)
    
    # Dự báo và chấm điểm trên tập Test
    preds = clf.predict(X_test)
    score = accuracy_score(y_test, preds)
    cv_scores.append(score)

mean_cv_score = np.mean(cv_scores)
baseline_acc = np.max(y.value_counts() / len(y))

print(f"-> Độ chính xác nền (Đoán bừa nhãn đa số): {baseline_acc*100:.1f}%")
print(f"-> Độ chính xác của AI (OOS Accuracy): {mean_cv_score*100:.1f}%")

# 4. Huấn luyện toàn bộ dữ liệu để lấy Feature Importances
clf.fit(X, y, sample_weight=w)

print("\n🌟 TOP 5 YẾU TỐ QUYẾT ĐỊNH GIÁ FPT (FEATURE IMPORTANCES):")
importances = pd.Series(clf.feature_importances_, index=X.columns).sort_values(ascending=False)
for col, imp in importances.head(5).items():
    print(f"{col:25}: {imp:.3f}")

🤖 BƯỚC 6: HUẤN LUYỆN MÔ HÌNH RANDOM FOREST

Đang chạy Cross-Validation (5-Fold, 4 Repeats)...
-> Độ chính xác nền (Đoán bừa nhãn đa số): 42.9%
-> Độ chính xác của AI (OOS Accuracy): 58.2%

🌟 TOP 5 YẾU TỐ QUYẾT ĐỊNH GIÁ FPT (FEATURE IMPORTANCES):
360_day_return           : 0.133
7_day_vol                : 0.115
30_day_vol               : 0.115
180_day_revenue_delta    : 0.114
360_day_revenue_delta    : 0.108
